In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-addon-utils
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay binuo gamit ang mga sumusunod na kinakailangan.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-addon-utils~=0.3.0
```
</details>

Ang package ng Qiskit addon utilities ay isang koleksyon ng mga functionality para suportahan ang mga workflow na gumagamit ng isa o higit pang Qiskit addon. Halimbawa, naglalaman ang package na ito ng mga function para sa paglikha ng mga Hamiltonian, pagbuo ng mga Trotter time-evolution circuit, at pagpipi at pagsasama ng mga quantum circuit.

## Pag-install

May dalawang paraan para i-install ang Qiskit addon utilities: PyPI at pagbuo mula sa source. Inirerekomenda na i-install ang mga package na ito sa isang [virtual environment](https://docs.python.org/3.10/tutorial/venv.html) para matiyak ang paghihiwalay ng mga dependencies ng package.

### I-install mula sa PyPI

Ang pinaka-simpleng paraan para i-install ang package ng Qiskit addon utilities ay sa pamamagitan ng PyPI.

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit import QuantumCircuit
from qiskit_addon_utils.coloring import auto_color_edges
from qiskit_addon_utils.slicing import combine_slices, slice_by_depth
from collections import defaultdict

backend = FakeSherbrooke()
coupling_map = backend.coupling_map

# Create naive circuit
circuit = QuantumCircuit(backend.num_qubits)
for edge in coupling_map.graph.edge_list():
    circuit.cz(edge[0], edge[1])


# Color the edges of the coupling map
coloring = auto_color_edges(coupling_map)
circuit_with_coloring = QuantumCircuit(backend.num_qubits)

# Make a reverse coloring dict in order to make the circuit
color_to_edge = defaultdict(list)
for edge, color in coloring.items():
    color_to_edge[color].append(edge)

# Place edges in order of color
for edges in color_to_edge.values():
    for edge in edges:
        circuit_with_coloring.cz(edge[0], edge[1])

print(f"The circuit without using edge coloring has depth: {circuit.depth()}")
print(
    f"The circuit using edge coloring has depth: {circuit_with_coloring.depth()}"
)

The circuit without using edge coloring has depth: 37
The circuit using edge coloring has depth: 3


### I-install mula sa source
<details>
<summary>
I-click dito para malaman kung paano i-install ang package na ito nang manu-mano.
</summary>

Kung gusto mong mag-ambag sa package na ito o nais itong i-install nang manu-mano, i-clone muna ang repository:

In [2]:
import numpy as np
from qiskit import QuantumCircuit

num_qubits = 9
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg" alt="Output of the previous code cell" />

at i-install ang package sa pamamagitan ng `pip`. Kung plano mong patakbuhin ang mga tutorial na nasa package repository, i-install din ang mga notebook dependency. Kung plano mong mag-develop sa repository, i-install ang mga `dev` na dependency.

In [3]:
# Slice circuit into partitions of depth 1
slices = slice_by_depth(qc, 1)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/6d824d97-edc6-4c88-bcfa-428731393f83-0.svg" alt="Output of the previous code cell" />

</details>

## Magsimula sa mga utility
May ilang mga module sa loob ng package na `qiskit-addon-utils`, kabilang ang isa para sa pagbuo ng mga problema para sa pag-simulate ng mga quantum system, graph coloring para mas mahusay na mailagay ang mga gate sa isang quantum circuit, at circuit slicing, na makakatulong sa [operator backpropagation](/guides/qiskit-addons-obp). Ang mga sumusunod na seksyon ay nagbubuod ng bawat module. Ang [dokumentasyon ng API](https://docs.quantum.ibm.com/api/qiskit-addon-utils) ng package ay naglalaman din ng kapaki-pakinabang na impormasyon.

### Pagbuo ng problema
Ang mga nilalaman ng module na [`qiskit_addon_utils.problem_generators`](../api/qiskit-addon-utils/problem-generators) ay kinabibilangan ng:

- Isang function na [`generate_xyz_hamiltonian()`](../api/qiskit-addon-utils/problem-generators#generate_xyz_hamiltonian), na bumubuo ng connectivity-aware na representasyong `SparsePauliOp` ng isang Ising-type XYZ model:

$$H = \sum_{(j,k)\in E} \left(J_x X_jX_k + J_yY_jY_k + J_zZ_jZ_k\right) + \sum_{j\in V} \left(h_x X_j + h_y Y_j + h_z Z_j\right) $$
- Isang function na [`generate_time_evolution_circuit()`](../api/qiskit-addon-utils/problem-generators#generate_time_evolution_circuit), na nagtatayo ng circuit na nagmo-modelo ng time evolution ng isang partikular na operator.
- Tatlong iba't ibang bagay na [`PauliOrderStrategy`](../api/qiskit-addon-utils/problem-generators#pauliorderstrategy) para sa pag-enumerate sa pagitan ng iba't ibang Pauli string ordering. Ito ay pangunahing kapaki-pakinabang kapag ginamit kasabay ng graph coloring at maaaring gamitin bilang mga argumento sa parehong function na `generate_xyz_hamiltonian()` at `generate_time_evolution_circuit()`.

### Graph coloring
Ang module na [`qiskit_addon_utils.coloring`](../api/qiskit-addon-utils/coloring) ay ginagamit para kulayan ang mga gilid sa isang coupling map at gamitin ang pagkukulay na ito para mas mahusay na mailagay ang mga gate sa isang quantum circuit. Ang layunin ng edge-colored coupling map na ito ay mahanap ang isang set ng mga kulay ng gilid na walang dalawang gilid ng parehong kulay ang nagbabahagi ng isang karaniwang node. Para sa isang QPU, nangangahulugan ito na ang mga gate sa magkatulad na may kulay na mga gilid (koneksyon ng qubit) ay maaaring patakbuhin nang sabay-sabay at mas mabilis na maisasagawa ang circuit.

Bilang isang mabilis na halimbawa, maaari mong gamitin ang function na [`auto_color_edges()`](../api/qiskit-addon-utils/coloring#auto_color_edges) para makabuo ng edge coloring para sa isang naive na circuit na nagpapatupad ng `CZGate` sa bawat koneksyon ng qubit. Ang code snippet sa ibaba ay gumagamit ng coupling map ng backend na `FakeSherbrooke`, lumilikha ng naive na circuit na ito, pagkatapos ay gumagamit ng function na `auto_color_edges()` para lumikha ng mas mahusay na katumbas na circuit.

In [4]:
from qiskit_addon_utils.slicing import slice_by_gate_types

slices = slice_by_gate_types(qc)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d011c56e-d741-491a-8867-54952cb7b757-0.svg" alt="Output of the previous code cell" />

If your workload is designed to exploit the physical qubit connectivity of the QPU it will be run on, you can create slices based on edge coloring. The code snippet below will assign a three-coloring to circuit edges and slice the circuit with respect to the edge coloring. (Note: this only affects non-local gates. Single-qubit gates will be sliced by gate type).

In [5]:
from qiskit_addon_utils.slicing import slice_by_coloring

# Assign a color to each set of connected qubits
coloring = {}
for i in range(num_qubits - 1):
    coloring[(i, i + 1)] = i % 3
coloring[(num_qubits - 1, 0)] = 2

# Create a circuit with operations added in order of color
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
edges = [
    edge for color in range(3) for edge in coloring if coloring[edge] == color
]
for edge in edges:
    qc.cx(edge[0], edge[1])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))

# Create slices by edge color
slices = slice_by_coloring(qc, coloring=coloring)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg" alt="Output of the previous code cell" />

### Slicing
Sa huli, ang module na [`qiskit-addon-utils.slicing`](../api/qiskit-addon-utils/slicing) ay naglalaman ng mga function at transpiler pass para sa paggawa ng mga "slice" ng circuit, na mga time-like na partisyon ng isang [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit) na sumasaklaw sa lahat ng qubit. Ang mga slice na ito ay pangunahing ginagamit para sa [operator backpropagation](/guides/qiskit-addons-obp-get-started). Ang pangunahing apat na paraan na maaaring i-slice ang isang circuit ay ayon sa uri ng gate, depth, coloring, o mga instruksyon ng [`Barrier`](../api/qiskit/circuit#barrier). Ang output ng mga slicing function na ito ay nagbabalik ng listahan ng mga bagay na [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit). Ang mga na-slice na circuit ay maaari ring pagsamahin muli gamit ang function na [`combine_slices()`](../api/qiskit-addon-utils/slicing#combine_slices). Basahin ang [API reference](../api/qiskit-addon-utils/slicing) ng module para sa karagdagang impormasyon.

Sa ibaba ay ilang mga halimbawa kung paano lumikha ng mga slice na ito gamit ang sumusunod na circuit:

In [6]:
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qc.barrier()
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.barrier()
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/5040a678-9d5f-4c8b-8a92-7de04c3fcfda-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg)

Sa kaso na walang malinaw na paraan para samantalahin ang istruktura ng isang circuit para sa operator backpropagation, maaari mong partisyonin ang circuit sa mga slice ng isang partikular na depth.

In [7]:
from qiskit_addon_utils.slicing import slice_by_barriers

slices = slice_by_barriers(qc)
slices[0].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/b1106c2f-711c-4d30-b91a-ea05f47598d8-0.svg" alt="Output of the previous code cell" />

In [8]:
slices[1].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/1afd2111-dd88-4e20-a137-f0c975e9d1bb-0.svg" alt="Output of the previous code cell" />

In [9]:
slices[2].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/360ab773-df81-4975-bb19-cd5f34e69b29-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg)

Kung mayroon kang custom na estratehiya sa pag-slice, maaari kang maglagay ng mga barrier sa circuit para tukuyin kung saan ito dapat i-slice at gamitin ang function na [`slice_by_barriers`](../api/qiskit-addon-utils/slicing#slice_by_barriers).